# Notebook #10: Niche Discovery

This notebook uses CellCharter to identify shared spatial immune niches across endometriosis lesions.

For v1, there are 2 peritoneal lesions.

**Main questions:**
1. What spatial niches exist within each lesion?
2. Are the same niches present in both lesions?
3. What cell types define each niche?

In [1]:
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    BASE_DIR = Path("/content/drive/MyDrive/endometriosis-spatial-immune-atlas-main")
except ImportError:
    BASE_DIR = Path("/Users/elenaeyre/Desktop/Science/spatial_tx/endometriosis-spatial-immune-atlas-main")

In [2]:
!pip install scanpy\
squidpy\
numpy==2.0.2\
pandas\
matplotlib\
seaborn\
cellcharter\
lightning\
scikit-learn\
anndata

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 11.7 MB/s  0:00:01eta 0:00:02m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [cellcharter] [cellcharter]


In [3]:
# -- Imports
from pathlib import Path

import scanpy as sc
import squidpy as sq

import numpy as np
import pandas as pd
import seaborn as sns
import cellcharter as cc
import matplotlib.pyplot as plt
from lightning.pytorch import seed_everything
from sklearn.preprocessing import StandardScaler
from matplotlib.patches import Patch

import anndata as ad
ad.settings.allow_write_nullable_strings = True

/Users/elenaeyre/Desktop/Science/Courses/Udacity/AI_4_Healthcare/Course_3/Project_Lesson_6/nd320-c1-emr-data-starter/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/elenaeyre/Desktop/Science/Courses/Udacity/AI_4_Healthcare/Course_3/Project_Lesson_6/nd320-c1-emr-data-starter/.venv/lib/python3.11/site-packages/docrep/decorators.py:43: SyntaxWarning: 'n_jobs' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)
/Users/elenaeyre/Desktop/Science/Courses/Udacity/AI_4_Healthcare/Course_3/Project_Lesson_6/nd320-c1-emr-data-starter/.venv/lib/python3.11/site-packages/docrep/decorators.py:43: SyntaxWarning: 'show_progress_bar' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)


In [4]:
# -- Paths
project_dir = BASE_DIR

input_path_346 = (
    project_dir
    / "data"
    / "interim"
    / "spatial"
    / "BEME_346_spatial_immunosenescence.h5ad"
)

input_path_355G = (
    project_dir
    / "data"
    / "interim"
    / "spatial"
    / "BEME_355G_spatial_immunosenescence.h5ad"
)

output_path_data = (
    project_dir
    / "data"
    / "interim"
    / "spatial"
)

output_path_figures = (
    project_dir
    / "figures"
    / "spatial"
    / "niche_discovery"
)

output_path_results = (
    project_dir
    / "results"
    / "spatial"
    / "niche_discovery"
)

output_path_data.mkdir(
    parents=True,
    exist_ok=True
)

output_path_figures.mkdir(
    parents=True,
    exist_ok=True
)

output_path_results.mkdir(
    parents=True,
    exist_ok=True
)

In [5]:
# -- SET THE SEED
seed_everything(3)

Seed set to 3


3

In [6]:
# -- Palette
niche_palette = {
    "Immune-depleted": "#D9D9D9",
    "Effector immune": "#A9643A",
    "Lymphoid-rich": "#D8B04C",
    "Immune hotspot": "#8E1F2F",
    "Immune-cold": "#A8BED1",
    "Adaptive-enriched": "#B45E82",
    "Diffuse immune": "#6F9272",
}

In [7]:
# -- Import objects
adata_346 = sc.read_h5ad(
    input_path_346
)

adata_355G = sc.read_h5ad(
    input_path_355G
)

adatas = {
    "BEME_346": adata_346,
    "BEME_355G": adata_355G
}

for sample_id, adata in adatas.items():
    print(
        sample_id,
        adata.shape
    )

BEME_346 (1388, 22243)
BEME_355G (1960, 22243)


Since the goal is to compare the neighborhoods between the tissue samples, the tissue objects will be grouped together to make sure that Niche 1 in 346 is the same niche in 355G. This also will make life easier if more tissues are added to the analysis pipeline.

In [8]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# FUNCTION_10_1 -- PREP AND CLEAN OBJECT BEFORE ANALYSIS
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def prepare_cellcharter_adata(
    adata,
    sample_id,
    abundance_key="q05_cell_abundance_w_sf"
):
    """Create a lightweight copy containing CellCharter inputs."""

    if abundance_key not in adata.obsm:
        raise KeyError(
            f"{abundance_key} is missing from {sample_id}."
        )

    adata_cc = adata.copy()

    abundance = (
        adata_cc.obsm[abundance_key]
        .copy()
    )

    # -- Remove cell2location prefix from labels
    abundance.columns = [
        column.split("_sf_", 1)[-1]
        for column in abundance.columns
    ]

    cell_types = abundance.columns.tolist()

    adata_cc.obsm[
        "immune_abundance"
    ] = abundance.to_numpy()

    adata_cc.uns[
        "immune_cell_types"
    ] = cell_types

    adata_cc.obs[
        "sample_id"
    ] = sample_id

    adata_cc.obs[
        "tissue_type"
    ] = "endometriosis"

    adata_cc.obs[
        "lesion_site"
    ] = "peritoneal"

    keep_obsm = {
        "spatial",
        "immune_abundance",
        abundance_key
    }

    for key in list(
        adata_cc.obsm.keys()
    ):
        if key not in keep_obsm:
            del adata_cc.obsm[key]

    return adata_cc


adatas_cc = {
    sample_id: prepare_cellcharter_adata(
        adata,
        sample_id=sample_id
    )
    for sample_id, adata
    in adatas.items()
}

cell_type_sets = {
    sample_id: tuple(
        adata.uns[
            "immune_cell_types"
        ]
    )
    for sample_id, adata
    in adatas_cc.items()
}

if len(
    set(
        cell_type_sets.values()
    )
) != 1:
    raise ValueError(
        "Spatial objects do not contain the same "
        "cell2location cell-type columns."
    )

cell_types = list(
    next(
        iter(
            cell_type_sets.values()
        )
    )
)

print(
    "Cell types:",
    cell_types
)

Cell types: ['B', 'CD4 T', 'CD8 T', 'DC', 'Mono-C', 'Mono-NC', 'NK-CD16+', 'NK-CD16-', 'TRM', 'Treg', 'nan', 'γδ T']


In [9]:
# -- Concat lesions

processed = []

spatial_uns = {}

for sample_id, adata in adatas_cc.items():

    sample = adata.copy()

    sample.obs_names_make_unique()

    sample.obs_names = [
        f"{sample_id}_{spot}"
        for spot in sample.obs_names
    ]

    sample.obs[
        "sample_id"
    ] = sample_id

    # -- Preserve sample-specific spatial metadata
    if "spatial" in sample.uns:
        spatial_uns.update(
            sample.uns["spatial"]
        )

    processed.append(
        sample
    )

adata_immune = ad.concat(
    processed,
    join="outer",
    merge="same",
    label=None,
    index_unique=None
)

adata_immune.uns[
    "spatial"
] = spatial_uns

adata_immune.obs[
    "sample_id"
] = adata_immune.obs[
    "sample_id"
].astype("category")

adata_immune.uns[
    "immune_cell_types"
] = cell_types

print(adata_immune)
print(
    adata_immune.obs[
        "sample_id"
    ].value_counts()
)

AnnData object with n_obs × n_vars = 3348 × 22243
    obs: 'in_tissue', 'array_row', 'array_col', 'library_id', 'sample_id', 'gsm_id', 'tissue_type', 'condition', 'lesion_site', '_indices', '_scvi_batch', '_scvi_labels', 'meanscell_abundance_w_sf_B', 'meanscell_abundance_w_sf_CD4 T', 'meanscell_abundance_w_sf_CD8 T', 'meanscell_abundance_w_sf_DC', 'meanscell_abundance_w_sf_Mono-C', 'meanscell_abundance_w_sf_Mono-NC', 'meanscell_abundance_w_sf_NK-CD16+', 'meanscell_abundance_w_sf_NK-CD16-', 'meanscell_abundance_w_sf_TRM', 'meanscell_abundance_w_sf_Treg', 'meanscell_abundance_w_sf_nan', 'meanscell_abundance_w_sf_γδ T', 'Total immune', 'B', 'CD4 T', 'CD8 T', 'DC', 'Mono-C', 'Mono-NC', 'NK-CD16+', 'NK-CD16-', 'TRM', 'Treg', 'nan', 'γδ T', 'leiden_0.2', 'leiden_0.4', 'leiden_0.6', 'leiden_0.8', 'region_cluster', 'total_immune_spot_level', 'total_immune_region_level', 'tissue_level_immune_class', 'projected_senescence_score', 'cluster_mean_senescence', 'projected_dysfunction_score', 'cluster

In [10]:
# -- Scale abundance & build neighborhoods
adata_immune.obsm[
    "immune_abundance_scaled"
] = StandardScaler().fit_transform(
    adata_immune.obsm[
        "immune_abundance"
    ]
)

sq.gr.spatial_neighbors(
    adata_immune,
    coord_type="grid",
    n_neighs=6,
    library_key="sample_id"
)

cc.gr.remove_long_links(
    adata_immune
)

cc.gr.aggregate_neighbors(
    adata_immune,
    use_rep="immune_abundance_scaled",
    n_layers=3
)

for key in adata_immune.obsm.keys():
    print(
        key,
        adata_immune.obsm[key].shape
    )

INFO     Creating graph using `None` transform and `2` libraries.                                                  


/var/folders/bh/3trj9gt94b517n0vny3w7ljh0000gn/T/ipykernel_17993/3658328562.py:10: FutureWarning: Calling `spatial_neighbors` is deprecated and will be removed in squidpy v1.9.0. Use `spatial_neighbors_knn`, `spatial_neighbors_radius`, `spatial_neighbors_delaunay`, `spatial_neighbors_grid`, or `spatial_neighbors_from_builder` instead.
  sq.gr.spatial_neighbors(
100%|██████████| 4/4 [00:00<00:00, 707.33it/s]

q05_cell_abundance_w_sf (3348, 12)
spatial (3348, 2)
immune_abundance (3348, 12)
immune_abundance_scaled (3348, 12)
X_cellcharter (3348, 48)


In [11]:
# -- CellCharter Niche Discovery
model = cc.tl.Cluster(
    n_clusters=7,
    random_state=3
)

model.fit(
    adata_immune,
    use_rep="X_cellcharter"
)

adata_immune.obs["immune_niche"] = model.predict(
    adata_immune,
    use_rep="X_cellcharter"
)

adata_immune.obs[
    "immune_niche"
] = adata_immune.obs[
    "immune_niche"
].astype("category")

print(
    adata_immune.obs[
        "immune_niche"
    ].value_counts()
)

display(
    pd.crosstab(
        adata_immune.obs[
            "immune_niche"
        ],
        adata_immune.obs[
            "sample_id"
        ]
    )
)

/Users/elenaeyre/Desktop/Science/Courses/Udacity/AI_4_Healthcare/Course_3/Project_Lesson_6/nd320-c1-emr-data-starter/.venv/lib/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


/Users/elenaeyre/Desktop/Science/Courses/Udacity/AI_4_Healthcare/Course_3/Project_Lesson_6/nd320-c1-emr-data-starte
r/.venv/lib/python3.11/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


immune_niche
2    748
1    587
0    573
3    561
4    478
6    316
5     85
Name: count, dtype: int64


sample_id,BEME_346,BEME_355G
immune_niche,,
0,561,12
1,0,587
2,748,0
3,77,484
4,2,476
5,0,85
6,0,316


In [12]:
# -- Composition tables
immune_df = pd.DataFrame(
    adata_immune.obsm[
        "immune_abundance"
    ],
    columns=cell_types,
    index=adata_immune.obs_names
)

immune_df[
    "immune_niche"
] = (
    adata_immune.obs[
        "immune_niche"
    ]
    .astype(str)
    .values
)

immune_df[
    "sample_id"
] = (
    adata_immune.obs[
        "sample_id"
    ]
    .astype(str)
    .values
)

niche_composition = (
    immune_df
    .groupby(
        "immune_niche"
    )[cell_types]
    .mean()
)

global_mean = (
    immune_df[
        cell_types
    ]
    .mean()
)

celltype_enrichment = (
    niche_composition
    .div(
        global_mean
    )
)

niche_counts = (
    adata_immune.obs[
        "immune_niche"
    ]
    .value_counts()
    .sort_index()
)

immune_state_niche = pd.crosstab(
    adata_immune.obs[
        "immune_niche"
    ],
    adata_immune.obs[
        "tissue_level_immune_class"
    ],
    normalize="index"
)

niche_sample = pd.crosstab(
    adata_immune.obs[
        "immune_niche"
    ],
    adata_immune.obs[
        "sample_id"
    ],
    normalize="columns"
)

display(niche_composition)
display(celltype_enrichment)
display(immune_state_niche)
display(niche_sample)

,B,CD4 T,CD8 T,DC,Mono-C,Mono-NC,NK-CD16+,NK-CD16-,TRM,Treg,nan,γδ T
immune_niche,,,,,,,,,,,,
0,1.528787,4.073627,1.525148,0.785043,0.452638,0.355328,12.871732,0.266639,0.420513,1.368898,0.326669,1.986477
1,5.627999,10.608083,6.925060,3.879352,2.271382,1.026709,35.437359,1.059574,0.823426,14.316958,0.709761,3.206573
2,6.623822,9.184680,6.749630,1.666100,2.152165,0.939642,41.815449,0.854383,0.823265,5.626564,0.126867,7.297832
3,0.135364,0.544447,0.154563,0.125495,0.045947,0.024302,1.494119,0.041167,0.029541,0.448562,0.014394,0.080460
4,1.142200,17.244190,1.152991,1.463824,0.317098,0.251545,15.398511,0.255720,0.780515,2.623940,0.118902,1.479706
5,18.465572,6.571492,22.649033,8.812722,7.818628,3.159049,94.166794,3.578301,1.162703,44.832493,1.695330,8.112494
6,8.484985,31.100159,9.828599,5.957807,2.948268,1.674763,57.591827,1.845857,1.747304,20.343174,0.652385,5.767131


,B,CD4 T,CD8 T,DC,Mono-C,Mono-NC,NK-CD16+,NK-CD16-,TRM,Treg,nan,γδ T
immune_niche,,,,,,,,,,,,
0,0.365416,0.396864,0.326139,0.356377,0.304545,0.487407,0.459151,0.364876,0.591361,0.182286,0.981874,0.566322
1,1.345224,1.033469,1.480862,1.761067,1.528235,1.408348,1.264094,1.449948,1.157970,1.906484,2.133344,0.914157
2,1.583250,0.894797,1.443348,0.756341,1.448023,1.288918,1.491609,1.169160,1.157744,0.749248,0.381328,2.080528
3,0.032355,0.053042,0.033052,0.056969,0.030914,0.033335,0.053297,0.056334,0.041542,0.059732,0.043266,0.022938
4,0.273013,1.679977,0.246557,0.664516,0.213351,0.345048,0.549284,0.349934,1.097626,0.349411,0.357388,0.421847
5,4.413707,0.640213,4.843291,4.000617,5.260541,4.333304,3.359046,4.896641,1.635090,5.970013,5.095687,2.312779
6,2.028111,3.029864,2.101757,2.704601,1.983658,2.297292,2.054372,2.525920,2.457205,2.708951,1.960888,1.644143


tissue_level_immune_class,Immune-high,Immune-low,Immune-moderate,Immune-moderate-high,Immune-moderate-low,Immune-poor
immune_niche,,,,,,
0,0.263525,0.317627,0.000000,0.160558,0.153578,0.104712
1,0.042589,0.155026,0.151618,0.117547,0.323680,0.209540
2,0.121658,0.221925,0.000000,0.001337,0.377005,0.278075
3,0.000000,0.206774,0.000000,0.130125,0.000000,0.663102
4,0.142259,0.414226,0.000000,0.012552,0.152720,0.278243
5,0.588235,0.058824,0.305882,0.023529,0.011765,0.011765
6,0.126582,0.341772,0.098101,0.253165,0.110759,0.069620


sample_id,BEME_346,BEME_355G
immune_niche,,
0,0.404179,0.006122
1,0.000000,0.299490
2,0.538905,0.000000
3,0.055476,0.246939
4,0.001441,0.242857
5,0.000000,0.043367
6,0.000000,0.161224


In [13]:
# -- Niche abundance
fig, ax = plt.subplots(
    figsize=(7, 4)
)

sns.barplot(
    x=niche_counts.index.astype(str),
    y=niche_counts.values,
    ax=ax
)

ax.set_xlabel(
    "Immune niche"
)

ax.set_ylabel(
    "Number of spots"
)

ax.set_title(
    "CellCharter niche abundance"
)

plt.tight_layout()

plt.savefig(
    output_path_figures
    / "10_niche_abundance.png",
    bbox_inches="tight",
    dpi=300
)

plt.close()

In [14]:
# -- Immune state composition by niche
fig, ax = plt.subplots(
    figsize=(9, 6)
)

sns.heatmap(
    immune_state_niche,
    cmap="viridis",
    annot=True,
    fmt=".2f",
    ax=ax
)

ax.set_xlabel(
    "Immune infiltration class"
)

ax.set_ylabel(
    "CellCharter niche"
)

ax.set_title(
    "Immune-class composition of spatial niches"
)

plt.tight_layout()

plt.savefig(
    output_path_figures
    / "10_niche_immune_class_composition.png",
    bbox_inches="tight",
    dpi=300
)

plt.close()

In [15]:
# -- Immune cell enrichment within niches
fig, ax = plt.subplots(
    figsize=(12, 7)
)

sns.heatmap(
    celltype_enrichment,
    cmap="coolwarm",
    center=1,
    ax=ax
)

ax.set_xlabel(
    "Immune cell type"
)

ax.set_ylabel(
    "CellCharter niche"
)

ax.set_title(
    "Immune cell enrichment within spatial niches"
)

plt.xticks(
    rotation=45,
    ha="right"
)

plt.tight_layout()

plt.savefig(
    output_path_figures
    / "10_niche_celltype_enrichment.png",
    bbox_inches="tight",
    dpi=300
)

plt.close()

In [16]:
# -- Niche distribution across lesions
fig, ax = plt.subplots(
    figsize=(7, 6)
)

sns.heatmap(
    niche_sample,
    cmap="viridis",
    annot=True,
    fmt=".2f",
    ax=ax
)

ax.set_xlabel(
    "Lesion"
)

ax.set_ylabel(
    "Immune niche"
)

ax.set_title(
    "Spatial niche distribution by lesion"
)

plt.tight_layout()

plt.savefig(
    output_path_figures
    / "10_niche_distribution_by_lesion.png",
    bbox_inches="tight",
    dpi=300
)

plt.close()

In [17]:
niche_composition.round(2)

,B,CD4 T,CD8 T,DC,Mono-C,Mono-NC,NK-CD16+,NK-CD16-,TRM,Treg,nan,γδ T
immune_niche,,,,,,,,,,,,
0,1.530000,4.07,1.53,0.79,0.45,0.36,12.870000,0.27,0.42,1.370000,0.33,1.99
1,5.630000,10.61,6.93,3.88,2.27,1.03,35.439999,1.06,0.82,14.320000,0.71,3.21
2,6.620000,9.18,6.75,1.67,2.15,0.94,41.820000,0.85,0.82,5.630000,0.13,7.30
3,0.140000,0.54,0.15,0.13,0.05,0.02,1.490000,0.04,0.03,0.450000,0.01,0.08
4,1.140000,17.24,1.15,1.46,0.32,0.25,15.400000,0.26,0.78,2.620000,0.12,1.48
5,18.469999,6.57,22.65,8.81,7.82,3.16,94.169998,3.58,1.16,44.830002,1.70,8.11
6,8.480000,31.10,9.83,5.96,2.95,1.67,57.590000,1.85,1.75,20.340000,0.65,5.77


### Manual niche interpretation

Going to review the abundace and renrichment tables before assigning biological tables. Because CellCharter cluster numbes can change after rerunning the model, this mapping should always be treated as a checkpoint.

In [18]:
niche_labels = {
    "0": "Immune-depleted",
    "1": "Effector immune",
    "2": "Lymphoid-rich",
    "3": "Immune hotspot",
    "4": "Immune-cold",
    "5": "Adaptive-enriched",
    "6": "Diffuse immune"
}


observed_niches = set(
    adata_immune.obs[
        "immune_niche"
    ]
    .astype(str)
    .unique()
)

mapped_niches = set(
    niche_labels
)

if observed_niches != mapped_niches:

    missing = sorted(
        observed_niches
        - mapped_niches
    )

    extra = sorted(
        mapped_niches
        - observed_niches
    )

    raise ValueError(
        "Manual niche map does not match the current CellCharter result. "
        f"Missing mappings: {missing}; extra mappings: {extra}"
    )

adata_immune.obs[
    "immune_niche_label"
] = (
    adata_immune.obs[
        "immune_niche"
    ]
    .astype(str)
    .map(
        niche_labels
    )
)

adata_immune.obs[
    "immune_niche_label"
] = adata_immune.obs[
    "immune_niche_label"
].astype("category")

print(
    adata_immune.obs[
        "immune_niche_label"
    ].value_counts()
)

immune_niche_label
Lymphoid-rich        748
Effector immune      587
Immune-depleted      573
Immune hotspot       561
Immune-cold          478
Diffuse immune       316
Adaptive-enriched     85
Name: count, dtype: int64


In [19]:
# -- Spatial mapping with labels


# -- Squidpy uses categorical labels in sorted order
squidpy_niche_order = sorted(
    adata_immune.obs[
        "immune_niche_label"
    ]
    .dropna()
    .astype(str)
    .unique()
)

adata_immune.uns[
    "immune_niche_label_colors"
] = [
    niche_palette[label]
    for label in squidpy_niche_order
]

sq.pl.spatial_scatter(
    adata_immune,
    color="immune_niche_label",
    library_key="sample_id",
    spatial_key="spatial",
    size=2,
    img=False,
    ncols=2,
    figsize=(6, 6),
    legend_loc=None,
    title=[
        "BEME_346 lesion",
        "BEME_355G lesion"
    ]
)

legend_elements = [
    Patch(
        facecolor=niche_palette[label],
        label=label
    )
    for label in squidpy_niche_order
]

plt.gcf().legend(
    handles=legend_elements,
    loc="center right",
    title="Immune niche",
    bbox_to_anchor=(1.12, 0.5)
)

plt.savefig(
    output_path_figures
    / "10_niche_discovery.png",
    bbox_inches="tight",
    dpi=300
)

plt.close()

In [20]:
# -- SAVE

output_file = (
    output_path_data
    / "10_immune_niches.h5ad"
)

adata_immune.write_h5ad(
    output_file
)

niche_key = pd.DataFrame(
    {
        "cluster": list(
            niche_labels.keys()
        ),
        "label": list(
            niche_labels.values()
        )
    }
)

niche_key.to_csv(
    output_path_results
    / "10_immune_niche_labels.csv",
    index=False
)

niche_composition.to_csv(
    output_path_results
    / "10_niche_mean_abundance.csv"
)

celltype_enrichment.to_csv(
    output_path_results
    / "10_niche_celltype_enrichment.csv"
)

immune_state_niche.to_csv(
    output_path_results
    / "10_niche_immune_class_composition.csv"
)

niche_sample.to_csv(
    output_path_results
    / "10_niche_distribution_by_lesion.csv"
)

saved = sc.read_h5ad(
    output_file,
    backed="r"
)

print(saved)

saved.file.close()

AnnData object with n_obs × n_vars = 3348 × 22243 backed at '/Users/elenaeyre/Desktop/Science/spatial_tx/endometriosis-spatial-immune-atlas-main/data/interim/spatial/10_immune_niches.h5ad'
    obs: 'in_tissue', 'array_row', 'array_col', 'library_id', 'sample_id', 'gsm_id', 'tissue_type', 'condition', 'lesion_site', '_indices', '_scvi_batch', '_scvi_labels', 'meanscell_abundance_w_sf_B', 'meanscell_abundance_w_sf_CD4 T', 'meanscell_abundance_w_sf_CD8 T', 'meanscell_abundance_w_sf_DC', 'meanscell_abundance_w_sf_Mono-C', 'meanscell_abundance_w_sf_Mono-NC', 'meanscell_abundance_w_sf_NK-CD16+', 'meanscell_abundance_w_sf_NK-CD16-', 'meanscell_abundance_w_sf_TRM', 'meanscell_abundance_w_sf_Treg', 'meanscell_abundance_w_sf_nan', 'meanscell_abundance_w_sf_γδ T', 'Total immune', 'B', 'CD4 T', 'CD8 T', 'DC', 'Mono-C', 'Mono-NC', 'NK-CD16+', 'NK-CD16-', 'TRM', 'Treg', 'nan', 'γδ T', 'leiden_0.2', 'leiden_0.4', 'leiden_0.6', 'leiden_0.8', 'region_cluster', 'total_immune_spot_level', 'total_immune_r

## Interpretation checkpoint

The niche labels should be based on the relative immune-cell enrichment profiles, not cluster number alone. Because only two lesions are included, lesion-specific abundance differences should be described as observations rather than generalized niche frequencies.

The joint CellCharter model allows the same niche identity to be compared across lesions, while the lesion-distribution table shows whether each niche is shared or sample-enriched.
